# BioNexus 2 — Write a lab adapter (the IAL contract)

Any lab — simulator, Opentrons robot, cloud lab — implements the same five
methods. Agents never see your adapter directly; the SafetyEngine gateway does.

In [ ]:
from bionexus.adapters.lab.simulator import SimulatorLab

class MyRobotLab:
    """Drop-in replacement for SimulatorLab — same contract, real hardware."""
    name = "my-robot"

    def __init__(self):
        self._sim = SimulatorLab(seed=1)   # delegate physics to the simulator

    def list_capabilities(self):
        return {"assays": ["binding", "cytotoxicity"], "bsl_max": 1}

    def submit_protocol(self, protocol):
        return self._sim.submit_protocol(protocol)   # translate + enqueue on your robot

    def get_status(self, run_id): return self._sim.get_status(run_id)
    def fetch_results(self, run_id): return self._sim.fetch_results(run_id)
    def cancel(self, run_id): self._sim.cancel(run_id)

Wrap it in the safety gateway and use it exactly like the simulator:

In [ ]:
from bionexus.core.safety import LabGateway, SafetyEngine
from bionexus.adapters.store.sqlite import SqliteProvenanceStore
from bionexus.core.interfaces import system_actor
from bionexus.core.contracts import (Envelope, ProtocolOp, ProtocolStep,
                                     ProtocolSubmission, ReagentSpec, SampleKind, SampleSpec)

gw = LabGateway(MyRobotLab(), SafetyEngine(), SqliteProvenanceStore(), system_actor())
protocol = ProtocolSubmission(
    envelope=Envelope(experiment_branch_id="nb2"),
    title="quick binding check", assay_type="binding",
    steps=[ProtocolStep(seq=1, op=ProtocolOp.dispense)],
    samples=[SampleSpec(sample_id="CCO", kind=SampleKind.compound, concentration="10uM")],
    reagents=[ReagentSpec(name="buffer", amount="1 mL", bsl_level=1)],
)
run_id, rejection = gw.submit(protocol)
result = gw.results(protocol.envelope, run_id)
print("rejected:", rejection, "| result:", result.measurements[0].value,
      result.measurements[0].unit)

Try adding a reagent with `bsl_level=3` and watch the SafetyEngine block it.